# MovieLens

## Descripción MovieLens

Los datasets de MovieLens fueron recolectados por 'GroupLens Research Project' en la Universidad de Minnesota.

Este conjunto de datos consta de:
* 100.000 valoraciones (en una escala de 1 a 5) realizadas por 943 usuarios sobre 1.682 películas.
* Cada usuario ha valorado al menos 20 películas.
    * Información demográfica básica de los usuarios (edad, género, ocupación, código postal).

Los datos se recopilaron a través del sitio web MovieLens (movielens.umn.edu) durante un periodo de siete meses, desde el 19 de septiembre de 1997 hasta el 22 de abril de 1998. Estos datos han sido depurados: se eliminaron del conjunto aquellos usuarios que tenían menos de 20 valoraciones o cuya información demográfica estaba incompleta.

---

Los datos están distribuidos en 6 conjuntos de datos:

**Data** (u.data)
* El conjunto de datos completo: 100 000 valoraciones realizadas por 943 usuarios sobre 1682 elementos. Cada usuario ha valorado al menos 20 películas.
* Los usuarios y los elementos están numerados consecutivamente a partir del 1.
* Los datos están ordenados aleatoriamente.
* Se trata de una lista separada por tabuladores con el formato: **user id | item id | rating | timestamp**.
* Las marcas de tiempo corresponden a segundos Unix contados desde el 1 de enero de 1970 (UTC).

**Info** (u.info)
* El número de usuarios, películas (items), y ratings en el conjunto de datos.

**Item** (u.item)
* Información sobre los elementos (películas)
* Se trata de una lista con los campos separados por tabuladores: **movie id | movie title | release date | video release date | IMDb URL | unknown | Action | Adventure | Animation | Children's | Comedy | Crime | Documentary | Drama | Fantasy | Film-Noir | Horror | Musical | Mystery | Romance | Sci-Fi | Thriller | War | Western**.
* Los últimos 19 campos corresponden a los géneros; un 1 indica que la película pertenece a ese género y un 0 indica que no; las películas pueden pertenecer a varios géneros simultáneamente.
* Los ID de las películas son los mismos que se utilizan en el conjunto de datos u.data.

**Genre** (u.genre)
* Una lista de los géneros.

**User** (u.user)
* Información demográfica sobre los usuarios.
* Se trata de una lista con los campos separados por tabuladores: **user id | age | gender | occupation | zip code**.
* Los ID de usuario son los mismos que se utilizan en el conjunto de datos u.data.

**Occupation** (u.occupation)
* Una lista de las ocupaciones.

## Importación de librerías y datos

In [37]:
import pandas as pd

In [38]:
data = pd.read_csv("../../data/raw/u.data", sep="\t", names=["user_id", "item_id", "rating", "timestamp"])
info = pd.read_csv("../../data/raw/u.info", sep=" ", names=["info", "value"])
item = pd.read_csv("../../data/raw/u.item", sep="|", encoding="latin1",
                   names=["movie id", "movie title", "release date", "video release date",
                          "IMDb URL", "unknown", "Action", "Adventure", "Animation",
                          "Children's", "Comedy", "Crime", "Documentary", "Drama",
                          "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",
                          "Romance", "Sci-Fi", "Thriller", "War", "Western"])
genre = pd.read_csv("../../data/raw/u.genre", sep="|", names=["genre", "value"])
user = pd.read_csv("../../data/raw/u.user", sep="|", names=["user_id", "age", "gender", "occupation", "zip_code"])
occupation = pd.read_csv("../../data/raw/u.occupation", sep="|", names=["occupation"])

In [39]:
# Copias de los dataframes originales
data_c = data.copy()
info_c = info.copy()
item_c = item.copy()
genre_c = genre.copy()
user_c = user.copy()
occupation_c = occupation.copy()

## Calidad y limpieza de los datos

Para cada tabla (data, info, item, genre, user, occupation) se hecha un vistazo y se inspeccionan valores faltantes y tipos por columna. Luego se revisan duplicados, que las fechas sean correctas, etc.

In [40]:
def print_shape_columns(df):
    print(f"Registros: {df.shape[0]}\nColumnas: {df.shape[1]}\n")
    print(f"Columnas: {df.columns}\n")

def print_missing_values(df):
    print(f"Porcentaje de valores faltantes: {df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100:.2f}%\n")
    print("Porcentaje de valores faltantes por columna:")
    display(df.isnull().sum() / (df.shape[0] * df.shape[1]) * 100)

### Data

In [41]:
# Vistazo a los datos
data.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [42]:
print_shape_columns(data)

Registros: 100000
Columnas: 4

Columnas: Index(['user_id', 'item_id', 'rating', 'timestamp'], dtype='str')



In [43]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    100000 non-null  int64
 1   item_id    100000 non-null  int64
 2   rating     100000 non-null  int64
 3   timestamp  100000 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB


Se ordena `data` de acuerdo a `user_id`, `item_id`, `timestamp`

In [44]:
data = data.sort_values(by=['user_id', 'item_id', 'timestamp'])

Se crea una nueva columna con la fecha y tiempo en que se hizo la reseña a partir de la columna de marca de tiempo ('timestamp')

In [45]:
data['datetime'] = pd.to_datetime(data['timestamp'], unit='s')
data.head()

,user_id,item_id,rating,timestamp,datetime
32236,1,1,5,874965758,1997-09-22 22:02:38
23171,1,2,3,876893171,1997-10-15 05:26:11
83307,1,3,4,878542960,1997-11-03 07:42:40
62631,1,4,3,876893119,1997-10-15 05:25:19
47638,1,5,3,889751712,1998-03-13 01:15:12


Se eliminan los siguientes registros:
* Sin calificación
* Con calificaciones fuera del rango [1, 5]
* Con un usuario no existente
* Con una película no existente
* Con parejas de `user_id` e `item_id` duplicadas

In [46]:
# Sin calificación
data = data[data['rating'].notnull()]

# Calificaciones fuera del rango
data = data[(data['rating'] >= 1) & (data['rating'] <= 5)]

# Con usuario no existente
data = data[data['user_id'].isin(user['user_id'])]

# Con película no existente
data = data[data['item_id'].isin(item['movie id'])]

# Parejas de usuario-película duplicadas
data = data.drop_duplicates(subset=['user_id', 'item_id'], keep='last')

In [47]:
print_missing_values(data)

Porcentaje de valores faltantes: 0.00%

Porcentaje de valores faltantes por columna:


user_id      0.0
item_id      0.0
rating       0.0
timestamp    0.0
datetime     0.0
dtype: float64

In [48]:
print(f"Cantidad de calificaciones: {data.shape[0]}")
print(f"Usuarios que dieron alguna calificación: {data['user_id'].unique().shape[0]}")
print(f"Películas calificadas: {data['item_id'].unique().shape[0]}")

Cantidad de calificaciones: 100000
Usuarios que dieron alguna calificación: 943
Películas calificadas: 1682


In [49]:
# Mínimo de calificaciones por usuario
user_item_counts = data.groupby('user_id')['item_id'].count()
print(f"Todos los usuarios han calificado por lo menos 20 películas: {(user_item_counts >= 20).all()}")

# Rango de fechas en que se hicieron las calificaciones
datetime = pd.to_datetime(data['timestamp'], unit='s')
print(f"Las calificaciones se hicieron entre {datetime.min()} y {datetime.max()}")

Todos los usuarios han calificado por lo menos 20 películas: True
Las calificaciones se hicieron entre 1997-09-20 03:05:10 y 1998-04-22 23:10:38


### Info

In [50]:
info

,info,value
0,943,users
1,1682,items
2,100000,ratings


In [51]:
info.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   info    3 non-null      int64
 1   value   3 non-null      str  
dtypes: int64(1), str(1)
memory usage: 180.0 bytes


### Item

In [52]:
# Vistazo de los datos
item.head()

,movie id,movie title,release date,video release date,IMDb URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [53]:
print_shape_columns(item)

Registros: 1682
Columnas: 24

Columnas: Index(['movie id', 'movie title', 'release date', 'video release date',
       'IMDb URL', 'unknown', 'Action', 'Adventure', 'Animation', 'Children's',
       'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
       'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western'],
      dtype='str')



In [54]:
item.info()

<class 'pandas.DataFrame'>
RangeIndex: 1682 entries, 0 to 1681
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   movie id            1682 non-null   int64  
 1   movie title         1682 non-null   str    
 2   release date        1681 non-null   str    
 3   video release date  0 non-null      float64
 4   IMDb URL            1679 non-null   str    
 5   unknown             1682 non-null   int64  
 6   Action              1682 non-null   int64  
 7   Adventure           1682 non-null   int64  
 8   Animation           1682 non-null   int64  
 9   Children's          1682 non-null   int64  
 10  Comedy              1682 non-null   int64  
 11  Crime               1682 non-null   int64  
 12  Documentary         1682 non-null   int64  
 13  Drama               1682 non-null   int64  
 14  Fantasy             1682 non-null   int64  
 15  Film-Noir           1682 non-null   int64  
 16  Horror           


* Se cambia el nombre de la columna `movie id` por `item_id`, el de `movie title` por `title`, `release date` por `release_date`.
* Se eliminan las columnas `video release date` y `IMDb URL` por no aportar no contener información relevante.

In [55]:
# Renombramiento de columnas
item = item.rename(columns={'movie id': 'item_id', 'movie title': 'title', 'release date': 'release_date'})

# Eliminación de columnas innecesarias
item = item.drop(columns=['video release date', 'IMDb URL'])

Se ordena `item` de acuerdo a `item_id`

In [56]:
item = item.sort_values(by=['item_id'])

Se cambia el tipo de dato de `release_date` a datetime para poder manipularlo más fácilmente

In [57]:
item['release_date'] = pd.to_datetime(item['release_date'], format='%d-%b-%Y', errors='coerce')

Se eliminan los siguientes registros:
* Sin ID de película
* Sin título de la película o título 'unknown'
* Con fecha de lanzamiento después de la fecha de finalización de recolección de los datos
* Sin pertenecer a algún género
* Con valor diferente de 0 o 1 para alguna columna de género

De `data` se eliminan las calificaciones de películas eliminadas en `item`

In [58]:
# Sin ID
item = item[item['item_id'].notnull()]

# Sin título o título 'unknown'
item = item[item['title'].notnull()]
item = item[item['title'] != 'unknown']

# Con fecha de lanzamiento luego de la recolección de datos
item = item[item['release_date'] <= data['datetime'].max()]

# Sin género
item = item[item.iloc[:, 3:].sum(axis=1) > 0]

# Con género inválido
item = item[item.iloc[:, 3:].isin([0, 1]).all(axis=1)]

# Eliminar también de data
data = data[data['item_id'].isin(item['item_id'])]

In [59]:
print_missing_values(item)

Porcentaje de valores faltantes: 0.00%

Porcentaje de valores faltantes por columna:


item_id         0.0
title           0.0
release_date    0.0
unknown         0.0
Action          0.0
Adventure       0.0
Animation       0.0
Children's      0.0
Comedy          0.0
Crime           0.0
Documentary     0.0
Drama           0.0
Fantasy         0.0
Film-Noir       0.0
Horror          0.0
Musical         0.0
Mystery         0.0
Romance         0.0
Sci-Fi          0.0
Thriller        0.0
War             0.0
Western         0.0
dtype: float64

Se eliminan los duplicados (mismo título de película), y se actualizan los IDs de las películas duplicadas en `data`.

In [60]:
# Actualización en data de IDs de películas duplicadas en item
item_copy = item.copy()
item_copy['id'] = item_copy['item_id']
movie_id_mapper = item_copy.set_index('id').groupby('title')['item_id'].transform('first')

data['item_id'] = data['item_id'].map(movie_id_mapper)
data['item_id'] = data['item_id'].astype(int)

# Eliminación de parejas usuario-película duplicadas en data tras la actualización
data = data.drop_duplicates(subset=['user_id', 'item_id'], keep='last')

# Eliminación de películas duplicadas en item
item = item.drop_duplicates(subset="title", keep="first")

Se eliminan las calificaciones con fecha previa a la fecha de lanzamiento de la película calificada

In [61]:
# Se unen 'item' y 'data' através del identificador de la película
data_item = data[['item_id', 'datetime']].merge(item[['item_id', 'release_date']], on='item_id', how='inner')

# Se resetea el índice y se eliminan las calificaciones inválidas
data = data.reset_index(drop=True)
data = data[data_item['datetime'] >= data_item['release_date']]

### Genre

In [62]:
genre

,genre,value
0,unknown,0
1,Action,1
2,Adventure,2
3,Animation,3
4,Children's,4
5,Comedy,5
6,Crime,6
7,Documentary,7
8,Drama,8
9,Fantasy,9


In [63]:
genre.info()

<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   genre   19 non-null     str  
 1   value   19 non-null     int64
dtypes: int64(1), str(1)
memory usage: 436.0 bytes


### User

In [64]:
user.head()

,user_id,age,gender,occupation,zip_code
0,1,24,M,technician,85711
1,2,53,F,other,94043
2,3,23,M,writer,32067
3,4,24,M,technician,43537
4,5,33,F,other,15213


In [65]:
print_shape_columns(user)

Registros: 943
Columnas: 5

Columnas: Index(['user_id', 'age', 'gender', 'occupation', 'zip_code'], dtype='str')



In [66]:
user.info()

<class 'pandas.DataFrame'>
RangeIndex: 943 entries, 0 to 942
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   user_id     943 non-null    int64
 1   age         943 non-null    int64
 2   gender      943 non-null    str  
 3   occupation  943 non-null    str  
 4   zip_code    943 non-null    str  
dtypes: int64(2), str(3)
memory usage: 37.0 KB


In [67]:
print_missing_values(user)

Porcentaje de valores faltantes: 0.00%

Porcentaje de valores faltantes por columna:


user_id       0.0
age           0.0
gender        0.0
occupation    0.0
zip_code      0.0
dtype: float64

Se ordena `user` de acuerdo a `user_id`

In [68]:
user = user.sort_values(by=['user_id'])

Se eliminan los siguientes registros:
* Con datos faltantes
* Con ID duplicado
* Con edad negativa o mayor que 110
* Con género diferente de 'M', 'm', 'F' o 'F'

In [69]:
# Con datos faltantes
user = user.dropna()

# Con ID duplicado
user = user.drop_duplicates(subset='user_id', keep='first')

# Con edad negativa o mayor que 110
user = user[(0 <= user['age']) & (user['age'] <= 110)]

# Solo M o F
user['gender'] = user['gender'].str.upper()
user = user[user['gender'].isin(['M', 'F'])]

# Eliminar también de data
data = data[data['user_id'].isin(user['user_id'])]

### Occupation

In [70]:
occupation

,occupation
0,administrator
1,artist
2,doctor
3,educator
4,engineer
5,entertainment
6,executive
7,healthcare
8,homemaker
9,lawyer


In [71]:
occupation.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   occupation  21 non-null     str  
dtypes: str(1)
memory usage: 300.0 bytes


## Resumen de datos procesados

### Data

In [72]:
print_shape_columns(data)

Registros: 99455
Columnas: 5

Columnas: Index(['user_id', 'item_id', 'rating', 'timestamp', 'datetime'], dtype='str')



In [73]:
print(f"Porcentaje de registros mantenidos: {data.shape[0]/data_c.shape[0] * 100:.3f}%")

Porcentaje de registros mantenidos: 99.455%


In [74]:
print_missing_values(data)

Porcentaje de valores faltantes: 0.00%

Porcentaje de valores faltantes por columna:


user_id      0.0
item_id      0.0
rating       0.0
timestamp    0.0
datetime     0.0
dtype: float64

### Item

In [75]:
print_shape_columns(item)

Registros: 1661
Columnas: 22

Columnas: Index(['item_id', 'title', 'release_date', 'unknown', 'Action', 'Adventure',
       'Animation', 'Children's', 'Comedy', 'Crime', 'Documentary', 'Drama',
       'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance',
       'Sci-Fi', 'Thriller', 'War', 'Western'],
      dtype='str')



In [76]:
print(f"Porcentaje de registros mantenidos: {item.shape[0]/item_c.shape[0] * 100:.3f}%")

Porcentaje de registros mantenidos: 98.751%


In [77]:
print_missing_values(item)

Porcentaje de valores faltantes: 0.00%

Porcentaje de valores faltantes por columna:


item_id         0.0
title           0.0
release_date    0.0
unknown         0.0
Action          0.0
Adventure       0.0
Animation       0.0
Children's      0.0
Comedy          0.0
Crime           0.0
Documentary     0.0
Drama           0.0
Fantasy         0.0
Film-Noir       0.0
Horror          0.0
Musical         0.0
Mystery         0.0
Romance         0.0
Sci-Fi          0.0
Thriller        0.0
War             0.0
Western         0.0
dtype: float64

### User

In [78]:
print_shape_columns(user)

Registros: 943
Columnas: 5

Columnas: Index(['user_id', 'age', 'gender', 'occupation', 'zip_code'], dtype='str')



In [79]:
print(f"Porcentaje de registros mantenidos: {user.shape[0]/user_c.shape[0] * 100:.3f}%")

Porcentaje de registros mantenidos: 100.000%


In [80]:
print_missing_values(user)

Porcentaje de valores faltantes: 0.00%

Porcentaje de valores faltantes por columna:


user_id       0.0
age           0.0
gender        0.0
occupation    0.0
zip_code      0.0
dtype: float64

## Exportación de datos procesados

In [ ]:
# data.to_csv("../../data/processed/data.csv", index=False)
# info.to_csv("../../data/processed/info.csv", index=False)
# item.to_csv("../../data/processed/item.csv", index=False)
# genre.to_csv("../../data/processed/genre.csv", index=False)
# user.to_csv("../../data/processed/user.csv", index=False)
# occupation.to_csv("../../data/processed/occupation.csv", index=False)

In [82]:
data_item = pd.merge(data, item, on="item_id")
data_user = pd.merge(data, user, on="user_id")
data_item_user = pd.merge(data_item, user, on="user_id")

In [ ]:
# data_item.to_csv("../../data/processed/data_item.csv", index=False)
# data_user.to_csv("../../data/processed/data_user.csv", index=False)
# data_item_user.to_csv("../../data/processed/data_item_user.csv", index=False)